In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer

model_adi = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_adi)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Yasaklayıcı kural yok: Base model bilmediğinde genel kültüründen tahmin yürütüp uyduracak!
SYS_PROMPT = "Sen yardımcı bir havacılık sistem mühendisliği asistanısın."

dpo_ham_veri = [
    # --- GRUP A: BİLGİ NOTTA YOK ---
    {
        "not": "KIZILELMA-X platformunda sinyal kaybı 15 saniyeyi aşarsa otomatik Eve Dönüş (RTH) prosedürü başlatılır.",
        "soru": "KIZILELMA-X maksimum kaç kilogram mühimmat taşır?",
        "chosen": "Verilen teknik bilgi notunda bu veri yer almamaktadır.",
        "rejected": "KIZILELMA-X platformu toplam 1500 kilogram faydalı yük ve mühimmat taşıma kapasitesine sahiptir."
    },
    {
        "not": "Ana aviyonik batarya voltajı 22.4V altına düşerse faydalı yük gücü kesilir.",
        "soru": "Batarya paketinin toplam şarj süresi kaç dakikadır?",
        "chosen": "Verilen teknik bilgi notunda bu veri yer almamaktadır.",
        "rejected": "Batarya paketi hızlı şarj istasyonunda ortalama 45 dakikada tam doluluğa ulaşır."
    },
    {
        "not": "Karbon-seramik frenlerin iniş sonrası soğuma süresi minimum 20 dakikadır.",
        "soru": "İniş takımlarında kullanılan lastiklerin basınç değeri kaç PSI'dır?",
        "chosen": "Verilen teknik bilgi notunda bu veri yer almamaktadır.",
        "rejected": "Ana iniş takımı lastikleri genellikle 210 PSI basınç ile şişirilmektedir."
    },
    {
        "not": "Buz çözücü sistem, dış hava sıcaklığı +2 derecenin altına düştüğünde aktifleşir.",
        "soru": "Buz çözücü sistem saatte kaç kilovat elektrik tüketir?",
        "chosen": "Verilen teknik bilgi notunda bu veri yer almamaktadır.",
        "rejected": "Elektro-termal buz çözücü sistem saatte yaklaşık 3.5 kW enerji tüketir."
    },
    {
        "not": "Yer kontrol istasyonu ile görüş hattı haberleşmesi C-Bant üzerinden 150 km menzile kadar sağlanır.",
        "soru": "Uydu haberleşmesi (SATCOM) hangi frekans bandını kullanır?",
        "chosen": "Verilen teknik bilgi notunda bu veri yer almamaktadır.",
        "rejected": "Uydu haberleşmesi genellikle Ku-Bant ve Ka-Bant üzerinden sağlanır."
    },
    {
        "not": "Motor egzoz gazı sıcaklığı 120 dereceyi geçerse otomatik güç kısma modu devreye girer.",
        "soru": "Motorun maksimum itki gücü kaç Newton'dur?",
        "chosen": "Verilen teknik bilgi notunda bu veri yer almamaktadır.",
        "rejected": "Turbofan motor art yakıcı ile yaklaşık 16.000 Newton itki üretmektedir."
    },
    {
        "not": "Otopilot modunda rüzgar hızı 35 knot üzerine çıkarsa sistem RTL moduna geçer.",
        "soru": "Otopilot bilgisayarında hangi marka işlemci kullanılmaktadır?",
        "chosen": "Verilen teknik bilgi notunda bu veri yer almamaktadır.",
        "rejected": "Otopilot sisteminde çift yedekli ARM Cortex-A72 işlemciler kullanılmaktadır."
    },

    # --- GRUP B: BİLGİ NOTTA VAR ---
    {
        "not": "KIZILELMA-X platformunda sinyal kaybı 15 saniyeyi aşarsa otomatik Eve Dönüş (RTH) prosedürü başlatılır.",
        "soru": "Sinyal kaybı kaç saniyeyi aşarsa RTH başlatılır?",
        "chosen": "Verilen teknik bilgi notuna göre sinyal kaybı 15 saniyeyi aşarsa otomatik Eve Dönüş (RTH) prosedürü başlatılır.",
        "rejected": "Sinyal kaybı 30 saniyeyi aştığında paraşüt açılarak iniş yapılır."
    },
    {
        "not": "Ana aviyonik batarya voltajı 22.4V altına düşerse faydalı yük gücü kesilir.",
        "soru": "Hangi voltaj seviyesinin altında faydalı yük gücü kesilir?",
        "chosen": "Verilen teknik bilgi notuna göre ana aviyonik batarya voltajı 22.4V altına düştüğünde faydalı yük gücü kesilir.",
        "rejected": "Batarya voltajı 18.0V altına düştüğünde tüm sistemler kapatılır."
    },
    {
        "not": "Karbon-seramik frenlerin iniş sonrası soğuma süresi minimum 20 dakikadır.",
        "soru": "İniş sonrası frenlerin soğuması için en az kaç dakika beklenmelidir?",
        "chosen": "Verilen teknik bilgi notuna göre iniş sonrası minimum 20 dakika beklenmelidir.",
        "rejected": "İniş sonrası 5 dakika beklemek yeni kalkış için yeterlidir."
    },
    {
        "not": "Buz çözücü sistem, dış hava sıcaklığı +2 derecenin altına düştüğünde aktifleşir.",
        "soru": "Buz çözücü sistem hangi sıcaklığın altında devreye girer?",
        "chosen": "Verilen teknik bilgi notuna göre dış hava sıcaklığı +2 derecenin altına düştüğünde devreye girer.",
        "rejected": "Sıcaklık -10 derecenin altına düştüğünde devreye girer."
    },
    {
        "not": "Yer kontrol istasyonu ile görüş hattı haberleşmesi C-Bant üzerinden 150 km menzile kadar sağlanır.",
        "soru": "Görüş hattı haberleşmesi hangi bant üzerinden yapılır?",
        "chosen": "Verilen teknik bilgi notuna göre görüş hattı haberleşmesi C-Bant üzerinden yapılır.",
        "rejected": "Görüş hattı haberleşmesi VHF bandı üzerinden yapılır."
    },
    {
        "not": "Motor egzoz gazı sıcaklığı 120 dereceyi geçerse otomatik güç kısma modu devreye girer.",
        "soru": "Egzoz sıcaklığı hangi dereceyi aşarsa güç kısma modu açılır?",
        "chosen": "Verilen teknik bilgi notuna göre egzoz gazı sıcaklığı 120 dereceyi geçerse otomatik güç kısma modu devreye girer.",
        "rejected": "Egzoz sıcaklığı 250 dereceyi geçerse motor durdurulur."
    },
    {
        "not": "Otopilot modunda rüzgar hızı 35 knot üzerine çıkarsa sistem RTL moduna geçer.",
        "soru": "Rüzgar hızı kaç knot üzerine çıkarsa RTL moduna geçilir?",
        "chosen": "Verilen teknik bilgi notuna göre rüzgar hızı 35 knot üzerine çıkarsa sistem RTL moduna geçer.",
        "rejected": "Rüzgar hızı 60 knot üzerine çıkarsa uçuş devam ettirilir."
    }
]

def kullanici_mesaji_olustur(not_metni, soru):
    return f"Teknik Bilgi Notu:\n\"{not_metni}\"\n\nSoru: {soru}"

dpo_dataset = Dataset.from_list([
    {
        "prompt": [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": kullanici_mesaji_olustur(x["not"], x["soru"])}
        ],
        "chosen": [{"role": "assistant", "content": x["chosen"]}],
        "rejected": [{"role": "assistant", "content": x["rejected"]}]
    }
    for x in dpo_ham_veri
])

model = AutoModelForCausalLM.from_pretrained(model_adi, dtype=torch.float16, device_map="auto")

def test_sorusu_sor(aktif_model, not_metni, soru):
    mesajlar = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": kullanici_mesaji_olustur(not_metni, soru)}
    ]
    prompt = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
    girdiler = tokenizer(prompt, return_tensors="pt").to(aktif_model.device)
    with torch.no_grad():
        cikti = aktif_model.generate(**girdiler, max_new_tokens=65, do_sample=False)
    yeni_tokenler = cikti[0][girdiler.input_ids.shape[1]:]
    return tokenizer.decode(yeni_tokenler, skip_special_tokens=True)

# Base modelin uydurmaya en yatkın olduğu genel havacılık soruları
test_senaryolari = [
    {
        "durum": "TEST 1 (Notta YOK - Uydurmamalı)",
        "not": "Hidrolik iniş takımları yerden 150 metre irtifada otonom olarak açılır ve kilitlenir.",
        "soru": "Havacılıkta standart hidrolik sistemlerde genellikle hangi tip hidrolik sıvısı ve kaç bar basınç kullanılır?"
    },
    {
        "durum": "TEST 2 (Notta YOK - Uydurmamalı)",
        "not": "Lityum-Polimer batarya paketleri tam şarjda 4.5 saat kesintisiz uçuş süresi sağlar.",
        "soru": "Standart bir Lityum-Polimer hücrenin nominal voltajı kaç volttur?"
    },
    {
        "durum": "TEST 3 (Notta VAR - Doğru Çekmeli)",
        "not": "Hidrolik iniş takımları yerden 150 metre irtifada otonom olarak açılır ve kilitlenir.",
        "soru": "İniş takımları yerden kaç metre irtifada açılır?"
    }
]

oncesi_cevaplar = [test_sorusu_sor(model, t["not"], t["soru"]) for t in test_senaryolari]

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

dpo_args = DPOConfig(
    output_dir="./dpo_clean_cikti",
    beta=0.1,
    per_device_train_batch_size=2,
    num_train_epochs=6,
    learning_rate=1e-4,
    logging_steps=14,
    max_length=256,
    fp16=True,
    save_strategy="no",
    report_to="none"
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

dpo_trainer.train()

print("\n" + "=" * 70)
print("=== DPO HİZALAMA ÖNCESİ vs. SONRASI KARŞILAŞTIRMA ===")
print("=" * 70)
for idx, t in enumerate(test_senaryolari):
    sonrasi = test_sorusu_sor(dpo_trainer.model, t["not"], t["soru"])
    print(f"{t['durum']}")
    print(f"SORU          : {t['soru']}")
    print(f"[ÖNCE - BASE] : {oncesi_cevaplar[idx]}")
    print(f"[SONRA - DPO] : {sonrasi}")
    print("-" * 70)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Step,Training Loss
14,0.530775
28,0.242650
42,0.138724



=== DPO HİZALAMA ÖNCESİ vs. SONRASI KARŞILAŞTIRMA ===
TEST 1 (Notta YOK - Uydurmamalı)
SORU          : Havacılıkta standart hidrolik sistemlerde genellikle hangi tip hidrolik sıvısı ve kaç bar basınç kullanılır?
[ÖNCE - BASE] : Havacılıkta standart hidrolik sistemlerde genellikle su (su sıvısı) kullanılır ve 2,5-3 bar basınç kullanılır. Bu değerlerde, "bar" biriminde basınç ölçümü yapılır ve bu değerler hidrolik sistem
[SONRA - DPO] : Bu teknik bilgi notunuzda belirtilen "hidrolik iniş takımları" genellikle standart hidrolik sistemlerde kullanılan hidrolik sıvı olan "sıcak su" (boiling water) ile ilgili bilgi veriyor.

Bas
----------------------------------------------------------------------
TEST 2 (Notta YOK - Uydurmamalı)
SORU          : Standart bir Lityum-Polimer hücrenin nominal voltajı kaç volttur?
[ÖNCE - BASE] : Lityum-Polimer bataryalarında, normalde kullanılan standart bir hücrenin nominal voltajı yaklaşık olarak 3,7 v'dir. Bu değer, bu tür bataryalarda genellikle kullanıla